# ⚡ Smart Grid Load Forecasting & Renewable Integration

## A Practical Energy Sector Data Science Project

**Problem Statement:** Modern power grids face severe challenges integrating intermittent renewable sources (solar/wind). Traditional load forecasting models fail to capture weather-dependent supply dynamics, causing supply-demand mismatches, increased curtailment, and costly fossil-fuel backup reliance.

**Solution:** Apply pandas time series analytics to build an intelligent forecasting pipeline that combines load data, weather conditions, and renewable generation patterns.

---
### Project Structure
| Part | Pandas Topic | Handbook Ref | Energy Application |
|------|-------------|--------------|-------------------|
| 1 | DataFrame/Series basics | 03.01 | Building utility meter data |
| 2 | Indexing & selection | 03.02 | Peak demand identification |
| 3 | Operations & ufuncs | 03.03 | Computing derived metrics |
| 4 | Missing values | 03.04 | Handling meter reading gaps |
| 5 | Hierarchical indexing | 03.05 | Multi-zonal grid analysis |
| 6 | Concat & append | 03.06 | Combining hourly/daily data |
| 7 | Merge & join | 03.07 | Enriching load with weather |
| 8 | Aggregation & grouping | 03.08 | Regional consumption patterns |
| 9 | Pivot tables | 03.09 | Executive grid dashboards |
| 10 | String operations | 03.10 | Cleaning zone/station names |
| 11 | Time series | 03.11 | Load curves & forecasting |
| 12 | Performance eval/query | 03.12 | Large-scale smart meter data |

---

## Setup — Imports & Configuration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)
pd.set_option('display.float_format', '{:.2f}'.format)

print(f'Pandas version: {pd.__version__}')
print(f'NumPy version:  {np.__version__}')

---
## Part 1: Foundations — Building Smart Grid DataFrames

**Reference:** *Introducing Pandas Objects* (03.01) — Series, DataFrame, Index

**Scenario:** We work at a regional utility managing 5 distribution zones with smart meters, solar farms, and wind turbines. Our goal is to predict load demand and balance renewable integration.

### 1.1 Create Master Zone Metadata Table

In [ ]:
# ── Build master zone infrastructure data (Chapter 03.01) ──
zone_data = {
    'zone':            ['Zone-A-Residential', 'Zone-B-Industrial', 'Zone-C-Commercial', 
                        'Zone-D-Mixed', 'Zone-E-Rural'],
    'grid_type':       ['Distribution', 'Transmission', 'Distribution', 'Distribution', 'Distribution'],
    'customer_count':  [125000, 450, 8500, 62000, 15000],
    'peak_demand_mw':  [450, 380, 290, 320, 180],
    'solar_capacity_mw': [50, 0, 30, 25, 15],
    'wind_capacity_mw':  [0, 50, 0, 20, 30]
}

df_zones = pd.DataFrame(zone_data)
df_zones

### 1.2 Generate Hourly Smart Meter Load Data (1 Week Sample)

Real utilities collect data every 15 minutes — we'll use hourly for demonstration.

In [ ]:
# ── Create 168 hours (1 week) of smart meter load data ──
np.random.seed(42)
hours = pd.date_range(start='2024-01-15 00:00', periods=168, freq='h')

# Pre-seeded realistic load patterns by zone
load_patterns = {
    'Zone-A-Residential': {
        'base_load_mw': 180,
        'morning_peak': [(6, 9), 1.4],
        'evening_peak': [(17, 21), 1.6],
        'weekend_mult': 0.85
    },
    'Zone-B-Industrial': {
        'base_load_mw': 320,
        'workday_mult': 1.25,
        'night_drop': 0.6
    },
    'Zone-C-Commercial': {
        'base_load_mw': 200,
        'business_hours': [(9, 17), 1.5],
        'weekend_mult': 0.4
    }
}

load_records = []
zones = df_zones['zone'].tolist()

for hour in hours:
    hour_of_day = hour.hour
    day_of_week = hour.dayofweek  # 0=Monday, 6=Sunday
    is_weekend = day_of_week >= 5
    
    for zone in zones:
        pattern = load_patterns.get(zone, {
            'base_load_mw': 100 + np.random.uniform(0, 50),
            'weekend_mult': 0.9 if is_weekend else 1.0
        })
        base = pattern['base_load_mw']
        multiplier = 1.0
        
        # Apply time-based patterns
        if 'morning_peak' in pattern:
            start, end = pattern['morning_peak'][0]
            mult = pattern['morning_peak'][1]
            if start <= hour_of_day <= end:
                multiplier *= mult
        if 'evening_peak' in pattern:
            start, end = pattern['evening_peak'][0]
            mult = pattern['evening_peak'][1]
            if start <= hour_of_day <= end:
                multiplier *= mult
        if 'business_hours' in pattern:
            start, end = pattern['business_hours'][0]
            mult = pattern['business_hours'][1]
            if not is_weekend and start <= hour_of_day <= end:
                multiplier *= mult
        if 'weekend_mult' in pattern:
            multiplier *= pattern['weekend_mult'] if is_weekend else 1.0
        if 'workday_mult' in pattern:
            multiplier *= pattern['workday_mult'] if not is_weekend else 1.0
        if 'night_drop' in pattern:
            if hour_of_day >= 22 or hour_of_day <= 5:
                multiplier *= pattern['night_drop']
        
        # Add realistic noise (+/- 15%)
        noise = np.random.uniform(0.85, 1.15)
        load = base * multiplier * noise
        
        load_records.append({
            'timestamp': timestamp,
            'hour_of_day': hour_of_day,
            'day_of_week': day_of_week,
            'is_weekend': is_weekend,
            'zone': zone,
            'load_mw': round(load, 2)
        })

df_load = pd.DataFrame(load_records)
print(f'Smart meter dataset: {len(df_load):,} records ({len(zones)} zones × 168 hours)')
df_load.head(24)

### 1.3 Generate Weather & Renewable Generation Data

Solar generation depends on irradiance and cloud cover. Wind depends on wind speed. Temperature affects AC/heating load.

In [ ]:
# ── Weather data correlated with time ──
weather_records = []

for i, hour in enumerate(hours):
    hour_of_day = hour.hour
    
    # Diurnal temperature pattern (colder at night, warmer midday)
    temp_base = 45 + 15 * np.sin((hour_of_day - 6) * np.pi / 12)
    temperature = temp_base + np.random.normal(0, 3)
    
    # Solar irradiance (zero at night, peaks noon)
    if 6 <= hour_of_day <= 18:
        solar_irradiance = max(0, 800 * np.sin((hour_of_day - 6) * np.pi / 12)) + np.random.normal(0, 50)
    else:
        solar_irradiance = 0
    
    # Wind speed (more variable)
    wind_speed = 8 + 5 * np.sin((hour_of_day + 3) * np.pi / 12) + np.random.normal(0, 2)
    wind_speed = max(0, wind_speed)
    
    # Cloud cover (affects solar)
    cloud_cover = np.random.randint(0, 100)
    
    weather_records.append({
        'timestamp': hour,
        'hour_of_day': hour_of_day,
        'temperature_f': round(temperature, 1),
        'solar_irradiance_wm2': round(max(0, solar_irradiance), 1),
        'wind_speed_mph': round(wind_speed, 1),
        'cloud_cover_pct': cloud_cover
    })

df_weather = pd.DataFrame(weather_records)
df_weather.head(12)

In [ ]:
# ── Calculate actual solar/wind generation based on weather ──
generation_records = []

for i, row in df_weather.iterrows():
    hr = row['hour_of_day']
    irr = row['solar_irradiance_wm2']
    wind = row['wind_speed_mph']
    clouds = row['cloud_cover_pct']
    
    for _, zone in df_zones.iterrows():
        z_name = zone['zone']
        
        # Solar generation (capacity factor affected by irradiance and cloud cover)
        solar_cap = zone['solar_capacity_mw']
        if solar_cap > 0 and irr > 0:
            cf_solar = min(irr / 1000, 1.0) * (1 - clouds / 150)  # Reduced by clouds
            solar_gen = solar_cap * cf_solar + np.random.uniform(-5, 5)
        else:
            solar_gen = 0
        
        # Wind generation (cut-in ~3 mph, rated ~25 mph)
        wind_cap = zone['wind_capacity_mw']
        if wind_cap > 0:
            if wind < 3:
                wind_gen = 0
            elif wind >= 25:
                wind_gen = wind_cap
            else:
                cf_wind = (wind - 3) / (25 - 3)
                wind_gen = wind_cap * cf_wind + np.random.uniform(-3, 3)
        else:
            wind_gen = 0
        
        generation_records.append({
            'timestamp': row['timestamp'],
            'hour_of_day': hr,
            'zone': z_name,
            'solar_generation_mw': round(max(0, solar_gen), 2),
            'wind_generation_mw': round(max(0, wind_gen), 2)
        })

df_generation = pd.DataFrame(generation_records)
df_generation.head(12)

> **Insight ▸** This setup mimics real utility architecture: load (consumption), weather, and renewable generation are tracked separately. Merging them later reflects production reality.

---
## Part 2: Intermediate — Indexing, Selection & Feature Engineering

**Reference:** *Data Indexing and Selection* (03.02), *Operating on Data* (03.03)

### 2.1 Indexing Patterns for Grid Analysis

In [ ]:
# ── Column selection ──
print('▼ Key load metrics:')
print(df_load[['timestamp', 'zone', 'load_mw']].head(12))

# ── Boolean masking: identify peak hours (>350 MW in any zone) ──
print('\n▼ Peak load hours (load > 350 MW):')
peaks = df_load[df_load['load_mw'] > 350]
print(peaks.to_string(index=False))

# ── Multi-condition filtering ──
print('\n▼ Weekend morning peaks in Zone-A:')
wknd_peaks = df_load[(df_load['zone']=='Zone-A-Residential') & 
                     (df_load['is_weekend']==True) & 
                     ((df_load['hour_of_day']>=6) & (df_load['hour_of_day']<=10))]
print(wknd_peaks[['timestamp','hour_of_day','load_mw']].to_string(index=False))

# ── loc for label-based access ──
print('\n▼ loc: Industrial zone (Zone-B) full week:')
industrial = df_load.loc[df_load['zone']=='Zone-B-Industrial', ['timestamp','hour_of_day','load_mw']]
print(industrial.head(16).to_string(index=False))

# ── Aggregate view by hour-of-day ──
print('\n▼ Average load by hour of day (all zones):')
avg_hourly = df_load.groupby('hour_of_day')['load_mw'].mean().round(2)
print(avg_hourly.to_string())

### 2.2 Operations — Compute Derived Grid Metrics

**Reference:** *Operating on Data in Pandas* (03.03) — UFuncs, index alignment

In [ ]:
# ── Merge load + generation + weather ──
df_grid = pd.merge(df_load, df_generation, on=['timestamp', 'zone'])
df_grid = pd.merge(df_grid, df_weather[['timestamp', 'temperature_f', 'solar_irradiance_wm2', 
                                         'wind_speed_mph', 'cloud_cover_pct']], on='timestamp')

# ── Net Load = Consumption - Renewable Generation (key metric for dispatch) ──
df_grid['net_load_mw'] = df_grid['load_mw'] - df_grid['solar_generation_mw'] - df_grid['wind_generation_mw']

# ── Renewable Penetration % ──
df_grid['renewable_pct'] = (df_grid['solar_generation_mw'] + df_grid['wind_generation_mw']) / df_grid['load_mw'] * 100

# ── Solar Curtailment Risk (when generation > load — rare but signals oversupply) ──
df_grid['curtailment_risk'] = (df_grid['solar_generation_mw'] > df_grid['load_mw']).astype(int)

# ── Temperature impact flag (extreme heat >85°F or cold <30°F increases AC/heating load) ──
df_grid['temp_stress'] = ((df_grid['temperature_f'] > 85) | (df_grid['temperature_f'] < 30)).astype(int)

print('Derived grid metrics computed: net_load_mw, renewable_pct, curtailment_risk, temp_stress')
df_grid[df_grid['zone']=='Zone-A-Residential'].head(24).to_string(index=False)

> **Insight ▸** **Net load** (demand minus renewables) is the critical metric for grid operators. It represents what conventional generation (natural gas, nuclear, hydro) must supply. High net load during evening peaks (when solar drops off) creates "duck curve" challenges.

---
## Part 3: Missing Data — Real-World Smart Meter Gaps

**Reference:** *Handling Missing Data* (03.04) — `isnull`, `dropna`, `fillna`

In [ ]:
# ── Simulate real-world meter reading failures ──
df_missing = df_grid.copy()

# Inject missing data patterns
missing_mask = [
    (df_missing['zone']=='Zone-C-Commercial') & (df_missing['hour_of_day']==14) & (df_missing['day_of_week']==2),
    (df_missing['zone']=='Zone-D-Mixed') & (df_missing['hour_of_day']==3),
    (df_missing['zone']=='Zone-E-Rural') & (df_missing['cloud_cover_pct']>80) & (df_missing['hour_of_day']>=10)
]
mask_combined = missing_mask[0] | missing_mask[1] | missing_mask[2]
df_missing.loc[mask_combined, ['load_mw', 'solar_generation_mw']] = np.nan

# Add some isolated failures
failures = [(50, 'load_mw'), (87, 'wind_generation_mw'), (142, 'temperature_f'), (155, 'solar_irradiance_wm2')]
for idx, col in failures:
    df_missing.loc[idx, col] = np.nan

print('▼ Missing value count by column:')
print(df_missing.isnull().sum())

# Strategy 1: Forward-fill for time series (standard utility practice)
df_ffill = df_missing.sort_values(['zone','timestamp']).copy()
df_ffill[['load_mw', 'solar_generation_mw', 'wind_generation_mw']] = (
    df_ffill.groupby('zone')[['load_mw', 'solar_generation_mw', 'wind_generation_mw']]
           .fillna(method='ffill').fillna(method='bfill')
)

# Strategy 2: Weather column imputation using hourly mean
for col in ['temperature_f', 'solar_irradiance_wm2', 'wind_speed_mph', 'cloud_cover_pct']:
    df_ffill[col] = df_ffill.groupby('hour_of_day')[col].transform(
        lambda s: s.fillna(s.mean())
    )

print(f'\nAfter imputation — remaining NaNs: {df_ffill.isnull().sum().sum()}')
print('\n▼ Imputed load values for Zone-C on Tuesday (were NaN):')
print(df_ffill[(df_ffill['zone']=='Zone-C-Commercial') & (df_ffill['day_of_week']==2)][['timestamp','load_mw']].to_string(index=False))

> **Insight ▸** Forward-fill (`ffill`) is preferred for smart meter data because consumption changes gradually. Dropping rows would lose valuable temporal continuity needed for forecasting models.

---
## Part 4: Advanced Integration — Merge, Concat, Hierarchical Indexing

**Reference:** *Hierarchical Indexing* (03.05), *Concat & Append* (03.06), *Merge & Join* (03.07)

### 4.1 Enrich Grid Data with External Forecasting Data

In [ ]:
# ── Simulated next-day weather forecast (external source) ──
forecast_data = pd.DataFrame({
    'forecast_hour': list(range(24)),
    'temp_forecast_f': [48, 47, 46, 45, 44, 45, 48, 52, 56, 60, 65, 68, 71, 72, 71, 69, 66, 63, 59, 56, 53, 51, 50, 49],
    'solar_forecast_wm2': [0,0,0,0,0,0,120,280,480,650,750,780,760,700,620,500,350,200,80,0,0,0,0,0],
    'wind_forecast_mph': [7, 6, 6, 5, 5, 6, 8, 10, 12, 14, 15, 14, 13, 12, 11, 10, 9, 8, 7, 6, 6, 7, 8, 7]
})

# ── Merge forecast into baseline for scenario analysis ──
df_scenario = pd.merge(df_grid[['hour_of_day', 'zone', 'load_mw', 'net_load_mw']].drop_duplicates(), 
                       forecast_data, left_on='hour_of_day', right_on='forecast_hour', how='left')

# Estimate next-day renewable generation with forecasted weather
df_scenario['estimated_solar_mw'] = (df_scenario['solar_forecast_wm2'] / 1000) * df_scenario.apply(
    lambda r: df_zones[df_zones['zone']==r['zone']]['solar_capacity_mw'].values[0] if len(df_zones[df_zones['zone']==r['zone']])>0 else 0,
    axis=1
)

df_scenario['estimated_wind_mw'] = df_scenario.apply(
    lambda r: 0 if r['wind_forecast_mph']<3 else 
              (r['wind_forecast_mph']-3)/(25-3) * df_zones[df_zones['zone']==r['zone']]['wind_capacity_mw'].values[0] if len(df_zones[df_zones['zone']==r['zone']])>0 else 0,
    axis=1
)

df_scenario['projected_net_load'] = df_scenario['load_mw'] - df_scenario['estimated_solar_mw'] - df_scenario['estimated_wind_mw']

print('▼ Next-day projection by hour (Zone-A example):')
proj_a = df_scenario[df_scenario['zone']=='Zone-A-Residential'][['forecast_hour','load_mw','estimated_solar_mw','projected_net_load']].round(2)
print(proj_a.to_string(index=False))

### 4.2 Concat — Stacking Multiple Weeks of Historical Data

In [ ]:
# ── Generate two additional weeks for trend analysis ──
weeks = []
for week_num in range(3):
    week_start = pd.Timestamp('2024-01-15') + pd.Timedelta(weeks=week_num)
    week_hours = pd.date_range(start=week_start, periods=168, freq='h')
    
    week_records = []
    for hour in week_hours:
        hour_of_day = hour.hour
        day_of_week = hour.dayofweek
        is_weekend = day_of_week >= 5
        
        for zone in zones:
            pattern = load_patterns.get(zone, {'base_load_mw': 100})
            base = pattern['base_load_mw']
            mult = pattern.get('weekend_mult', 1.0) if is_weekend else 1.0
            if 'morning_peak' in pattern:
                s,e = pattern['morning_peak'][0]
                if s <= hour_of_day <= e: mult *= pattern['morning_peak'][1]
            if 'evening_peak' in pattern:
                s,e = pattern['evening_peak'][0]
                if s <= hour_of_day <= e: mult *= pattern['evening_peak'][1]
            
            noise = np.random.uniform(0.85, 1.15)
            week_records.append({
                'week_number': week_num,
                'timestamp': hour,
                'hour_of_day': hour_of_day,
                'zone': zone,
                'load_mw': round(base * mult * noise, 2)
            })
    
    weeks.append(pd.DataFrame(week_records))

# Concatenate all weeks
df_multi_week = pd.concat(weeks, ignore_index=True)
df_multi_week['week_label'] = df_multi_week['week_number'].map({0:'Week-1', 1:'Week-2', 2:'Week-3'})

print(f'Multi-week dataset: {len(df_multi_week):,} records across {df_multi_week["week_number"].nunique()} weeks')
df_multi_week.head(30)

### 4.3 Hierarchical (MultiIndex) — Zonal + Temporal Analysis

In [ ]:
# ── Set MultiIndex for efficient zonal-time slicing ──
df_hier = df_multi_week.set_index(['zone', 'week_number', 'hour_of_day']).sort_index()

print('▼ MultiIndex DataFrame (Zone → Week → Hour):')
print(df_hier.head(36))

# ── Partial slice: all hours for one zone ──
print('\n▼ Partial slice — Zone-B (Industrial) full profile:')
zone_b = df_hier.loc['Zone-B-Industrial']
print(zone_b.head(24))

# ── Cross-section: all zones for Week 2 ──
print('\n▼ Cross-section — Week 2 across all zones (first 20 hours):')
week2 = df_hier.xs(1, level='week_number')
print(week2.head(20))

# ── Aggregation at different index levels ──
print('\n▼ Mean load by zone (level=0):')
print(df_hier.groupby(level=0)['load_mw'].mean().round(2).sort_values(ascending=False).to_string())

print('\n▼ Mean load by hour-of-day (level=2):')
print(df_hier.groupby(level=2)['load_mw'].mean().round(2).sort_index().to_string())

# ── Swap levels for alternative view ──
print('\n▼ Swapped index (Hour → Zone), Hour 18 (evening peak):')
swapped = df_hier.swaplevel(0, 2).sort_index()
print(swapped.xs(18, level='hour_of_day').groupby(level=0).mean().round(2).to_string())

> **Insight ▸** MultiIndex enables complex queries like "show me all industrial zones' loads during evening peak hours across all weeks" without slow iteration. This is essential for grid optimization at scale.

---
## Part 5: Aggregation, Grouping & Pivot Tables

**Reference:** *Aggregation and Grouping* (03.08), *Pivot Tables* (03.09)

### 5.1 GroupBy — Regional Consumption Patterns

In [ ]:
df_analysis = df_grid.copy()
df_analysis['grid_category'] = df_analysis['zone'].apply(lambda z: 'Urban' if z in ['Zone-A-Residential','Zone-C-Commercial','Zone-D-Mixed'] else 'Rural')

# ── Multi-aggregate by grid category ──
print('▼ Grid Category Summary:')
cat_agg = df_analysis.groupby('grid_category').agg({
    'load_mw': ['mean','std','min','max','sum'],
    'net_load_mw': ['mean','std'],
    'renewable_pct': ['mean','max'],
    'solar_generation_mw': 'sum'
}).round(2)
print(cat_agg.to_string())

# ── Hour-by-hour load curve ──
print('\n▼ Hourly Load Curve (all zones):')
hour_curve = df_analysis.groupby('hour_of_day').agg({
    'load_mw': ['mean', 'std'],
    'solar_generation_mw': 'mean',
    'wind_generation_mw': 'mean',
    'net_load_mw': 'mean'
}).round(2)
print(hour_curve.to_string())

# ── Zone ranking by average net load (what conventional generation must supply) ──
print('\n▼ Zones ranked by average net load (higher = more conventional gen needed):')
zone_rank = df_analysis.groupby('zone')['net_load_mw'].mean().sort_values(ascending=False).round(2)
print(zone_rank.to_string())

# ── Filter: hours with high renewable penetration (>40%) ──
high_renew = df_analysis[df_analysis['renewable_pct'] > 40]
print(f'\n▼ Hours with renewable penetration > 40%: {len(high_renew)} out of {len(df_analysis)} ({len(high_renew)/len(df_analysis)*100:.1f}%)')
print(high_renew.groupby('zone').size())

# ── Transform: load deviation from zone average ──
df_analysis['load_deviation'] = df_analysis.groupby('zone')['load_mw'].transform(
    lambda x: x - x.mean()
)
print('\n▼ Load deviation from mean (Zone-A sample):')
ua_sample = df_analysis[df_analysis['zone']=='Zone-A-Residential'][['timestamp','load_mw','load_deviation']].head(24)
print(ua_sample.to_string(index=False))

### 5.2 Pivot Tables — Executive Grid Dashboards

In [ ]:
# ── Load matrix: zone × hour ──
pivot_load = df_analysis.pivot_table(
    values='load_mw', index='zone', columns='hour_of_day',
    aggfunc='mean', margins=True, margins_name='Avg'
)
print('▼ Hourly Load Matrix (MW) — Zone × Hour:')
print(pivot_load.round(1).to_string())

# ── Net load by day of week × zone ──
pivot_net = df_analysis.pivot_table(
    values='net_load_mw', index='zone', columns='day_of_week',
    aggfunc='mean', margins=True, margins_name='Weekly Avg'
)
day_names = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']
pivot_net.columns = [day_names[c] for c in pivot_net.columns]
print('\n▼ Net Load (MW) — Zone × Day of Week:')
print(pivot_net.round(1).to_string())

# ── Renewable generation by hour of day and renewable type ──
df_renew = df_analysis.melt(
    id_vars=['hour_of_day', 'zone'],
    value_vars=['solar_generation_mw', 'wind_generation_mw'],
    var_name='gen_type', value_name='mw'
).groupby(['hour_of_day', 'gen_type'])['mw'].mean().unstack()

df_renew.index.name = 'hour_of_day'
print('\n▼ Avg Renewable Generation by Hour:')
print(df_renew.round(1).to_string())

# ── Duck curve visualization data ──
#    Duck curve: net load vs time shows steep afternoon drop (solar kicks in) + steep evening rise
duck_curve = df_analysis.groupby('hour_of_day')['net_load_mw'].mean().reset_index()
print('\n▼ Duck Curve Data (net load over 24h):')
print(duck_curve.to_string(index=False))

> **Insight ▸** The **Duck Curve** is the signature challenge of solar-rich grids: net load drops sharply in mid-afternoon (solar peaks), then rises steeply after sunset (solar vanishes, demand remains). This creates ramping stress on conventional generators. Pandas pivot tables extract this pattern directly from raw meter data.

---
## Part 6: String Operations — Cleaning Messy Station Names

**Reference:** *Vectorized String Operations* (03.10) — `.str` accessor

In [ ]:
# ── Simulate dirty station/meter IDs from multiple vendors ──
dirty_stations = pd.DataFrame({
    'raw_station_id': ['ZONE-A-METER-001', 'zone-b-meter-025', 'ZoneC_MTR_015 ', 'ZONE-D-METER-089', 
                       'zone_e_meter_112', 'Zone-A-Meter-002', 'ZONE_C_MTR_030', 'zone-d-Mtr-101'],
    'raw_zone_type': ['RESIDENTIAL', 'residential ', 'Commercial', ' MIXED ', 'rural', 'Residential', 'COMMERICAL', 'Mixed']
})

# 1. Standardize casing and strip whitespace
dirty_stations['station_id'] = dirty_stations['raw_station_id'].str.strip().str.upper()
dirty_stations['zone_type'] = dirty_stations['raw_zone_type'].str.strip().str.title()

# 2. Fix typos (simple regex replacement)
dirty_stations['zone_type'] = dirty_stations['zone_type'].replace({'Commerical':'Commercial'})

# 3. Extract zone letter via pattern matching
dirty_stations['zone_letter'] = dirty_stations['station_id'].str.extract('(ZONE-(.)-METER')[1]

# 4. Parse meter number
dirty_stations['meter_number'] = dirty_stations['station_id'].str.extract('-([0-9]+)$')[0].astype(int)

print('▼ Standardized station metadata:')
print(dirty_stations[['raw_station_id','station_id','zone_letter','meter_number','zone_type']].to_string(index=False))

# 5. Validate station naming consistency
invalid = dirty_stations[
    (~dirty_stations['station_id'].str.startswith('ZONE')) | 
    (~dirty_stations['station_id'].str.contains('-METER'))
]
print(f'\n▼ Non-compliant station IDs: {len(invalid)}')
if len(invalid) > 0:
    print(invalid['raw_station_id'].tolist())

---
## Part 7: Time Series — Load Curves & Forecasting Features

**Reference:** *Working with Time Series* (03.11) — DatetimeIndex, rolling, shifting

In [ ]:
# ── Convert to proper DatetimeIndex ──
df_ts = df_multi_week.copy()
df_ts = df_ts.set_index('timestamp').sort_index()

print('▼ Time-indexed load data (sample):')
print(df_ts[df_ts['zone']=='Zone-A-Residential'].head(48).to_string())

# ── Rolling window features: 6-hour moving average smooths volatility ──
df_ts['load_ma6'] = df_ts.groupby('zone')['load_mw'].transform(
    lambda x: x.rolling(window=6, min_periods=1).mean()
)

print('\n▼ 6-Hour Moving Average Load (Zone-D Mixed):')
zd = df_ts[df_ts['zone']=='Zone-D-Mixed'][['load_mw','load_ma6']].head(36)
print(zd.to_string())

# ── Lag features for ML forecasting (t-1hr, t-24hr, t-1week) ──
df_ts['load_lag_1hr'] = df_ts.groupby('zone')['load_mw'].shift(1)
df_ts['load_lag_24hr'] = df_ts.groupby('zone')['load_mw'].shift(24)
df_ts['load_lag_168hr'] = df_ts.groupby('zone')['load_mw'].shift(168)  # 1 week ago

print('\n▼ Lagged load features (Zone-C Commercial):')
zc = df_ts[df_ts['zone']=='Zone-C-Commercial'][['load_mw','load_lag_1hr','load_lag_24hr','load_lag_168hr']].head(30)
print(zc.to_string())

# ── Difference features (rate of change) ──
df_ts['load_diff_1hr'] = df_ts.groupby('zone')['load_mw'].diff(1)
df_ts['load_diff_24hr'] = df_ts.groupby('zone')['load_mw'].diff(24)

print('\n▼ Rate of change features (Zone-E Rural):')
ze = df_ts[df_ts['zone']=='Zone-E-Rural'][['load_mw','load_diff_1hr','load_diff_24hr']].head(30)
print(ze.to_string())

# ── Isolate weekdays vs weekends for separate modeling ──
weekday_load = df_ts[df_ts['day_of_week']<5].groupby('hour_of_day')['load_mw'].mean()
weekend_load = df_ts[df_ts['day_of_week']>=5].groupby('hour_of_day')['load_mw'].mean()

print('\n▼ Weekday vs Weekend Load Profiles (all zones averaged):')
profile = pd.DataFrame({'Weekday': weekday_load, 'Weekend': weekend_load})
print(profile.round(2).to_string())

# Save duck curve data for visualization
duck_df = df_ts.groupby('hour_of_day')['net_load_mw'].mean().reset_index()

> **Insight ▸** Lag features (`shift()`) are the foundation of autoregressive forecasting models (ARIMA, LSTM). Separating weekday/weekend profiles accounts for fundamentally different load patterns — residential peaks differ dramatically between weekdays (workday mornings) and weekends (late mornings).

---
## Part 8: Performance — Large-Scale Smart Meter Processing

**Reference:** *High-Performance Pandas* (03.12) — `eval()`, `query()`

In [ ]:
# ── Simulate large-scale dataset (50,000 smart meter readings) ──
np.random.seed(99)
N = 50_000
big_meters = pd.DataFrame({
    'meter_id': np.random.choice(range(1000), N),
    'timestamp': np.random.choice(hours, N),
    'zone': np.random.choice(zones, N),
    'load_kwh': np.random.uniform(0.5, 5.0, N),
    'temperature_f': np.random.uniform(30, 95, N),
    'hour_of_day': np.random.choice(range(24), N),
    'day_of_week': np.random.choice(range(7), N)
})
big_meters['timestamp'] = pd.to_datetime(big_meters['timestamp'])

print(f'Dataset size: {len(big_meters):,} rows | Memory: {big_meters.memory_usage(deep=True).sum()/1e6:.1f} MB')

# ── Traditional arithmetic ──
big_meters['load_kwh_norm'] = big_meters['load_kwh'] / big_meters.groupby('meter_id')['load_kwh'].transform('mean')
big_meters['temp_band'] = np.where(big_meters['temperature_f']<40,'Cold',
                              np.where(big_meters['temperature_f']<75,'Comfort','Hot'))

# ── eval() — efficient multi-column computation ──
big_eval = big_meters.drop(columns=['load_kwh_norm','temp_band']).copy()
big_eval.eval('''
    load_kwh_norm = load_kwh / groupby('meter_id')['load_kwh'].transform('mean')
    temp_band = np.where(temperature_f<40,"Cold",np.where(temperature_f<75,"Comfort","Hot"))
''', inplace=True)

print('\\n▼ eval() computed columns — sample:')
print(big_eval.head(10).to_string())

# ── query() — efficient filtering ──
threshold = 3.5
result = big_meters.query('load_kwh > @threshold and temp_band=="Hot"')
print(f'\\n▼ query() filter: load > {threshold} kWh AND hot temperatures')
print(f'Matched: {len(result):,} records')
print(result.groupby('zone').size())

# ── Timing comparison ──
import timeit
trad_time = timeit.timeit(
    lambda: big_meters[(big_meters['load_kwh']>3.5) & (big_meters['temp_band']=='Hot')],
    number=50
)
query_time = timeit.timeit(
    lambda: big_meters.query('load_kwh > 3.5 and temp_band=="Hot"'),
    number=50
)
print(f'\\n▼ Timing (50 iterations):')
print(f'  Boolean mask: {trad_time:.4f}s')
print(f'  query():      {query_time:.4f}s')
print(f'  Speedup:      {trad_time/query_time:.2f}×')

---
## Part 9: Synthesis — Grid Operations Dashboard

In [ ]:
def grid_operations_dashboard(df):
    '''Complete grid analysis pipeline producing operator-ready insights.'''
    sep = '=' * 70
    print(sep)
    print('  ⚡ REGIONAL GRID OPERATIONS DASHBOARD ⚡')
    print(sep)

    # 1. Overview
    print(f'\n[1] SYSTEM OVERVIEW')
    print(f'  Zones monitored: {df["zone"].nunique()}  |  Hours analyzed: {df["hour_of_day"].nunique()}  |  Total records: {len(df):,}')

    # 2. Peak load detection
    print(f'\n[2] PEAK LOAD EVENTS')
    peak_records = df.nlargest(10, 'load_mw')[['zone','timestamp','load_mw','hour_of_day']]
    print(peak_records.to_string(index=False))

    # 3. Renewable integration summary
    print(f'\n[3] RENEWABLE INTEGRATION METRICS')
    renew_metrics = df.groupby('zone').agg({
        'solar_generation_mw': 'sum',
        'wind_generation_mw': 'sum',
        'renewable_pct': 'mean'
    }).round(2)
    print(renew_metrics.to_string())

    # 4. Net load (conventional dispatch requirement)
    print(f'\n[4] NET LOAD BY ZONE (MW — what conventional gen must supply)')
    net_by_zone = df.groupby('zone')['net_load_mw'].mean().sort_values(ascending=False).round(2)
    for z, v in net_by_zone.items():
        bar = '█' * int(min(v/20, 20))
        print(f'  {z:20s} {v:8.2f}  {bar}')

    # 5. Duck curve analysis
    print(f'\n[5] DUCK CURVE ANALYSIS')
    duck = df.groupby('hour_of_day')['net_load_mw'].mean()
    duck_min = duck.min()
    duck_max = duck.max()
    duck_ramp = duck.loc[17:20].mean() - duck.loc[13:16].mean()  # Evening ramp
    print(f'  Lowest net load (midday solar peak): {duck_min:.2f} MW at hour {duck.idxmin()}')
    print(f'  Highest net load (evening peak):     {duck_max:.2f} MW at hour {duck.idxmax()}')
    print(f'  Evening ramp rate (hrs 13→20):      {duck_ramp:.2f} MW/hour')

    # 6. Curtailment risk hours
    print(f'\n[6] CURTAILMENT RISK ASSESSMENT')
    risk = df[df['curtailment_risk']==1]
    print(f'  Hours with solar > load (oversupply risk): {len(risk)}')
    if len(risk) > 0:
        print(risk.groupby(['zone','hour_of_day']).size())
    else:
        print('  No curtailment risk detected in analysis period.')

    # 7. Temperature stress correlation
    print(f'\n[7] TEMPERATURE STRESS IMPACT ON LOAD')
    stress_load = df[df['temp_stress']==1]['load_mw'].mean()
    normal_load = df[df['temp_stress']==0]['load_mw'].mean()
    uplift = (stress_load - normal_load) / normal_load * 100
    print(f'  Average load during temp stress:  {stress_load:.2f} MW')
    print(f'  Average load during comfort:      {normal_load:.2f} MW')
    print(f'  Load uplift due to extreme temps: +{uplift:.1f}%')

    print(f'\n{sep}')
    print('  Analysis complete. Export-ready for grid dispatch planning.')
    print(sep)

    return {
        'peak_events': peak_records,
        'renewable_metrics': renew_metrics,
        'net_load': net_by_zone,
        'duck_curve': duck
    }

# Run the full dashboard
grid_dashboard = grid_operations_dashboard(df_grid)

---
## Part 10: Export — Deliverables for Grid Operators

In [ ]:
# ── Prepare final dataset for export ──
export_cols = ['timestamp','zone','hour_of_day','day_of_week','load_mw','solar_generation_mw',
               'wind_generation_mw','net_load_mw','renewable_pct','temperature_f',
               'solar_irradiance_wm2','wind_speed_mph','cloud_cover_pct','curtailment_risk']

final_export = df_grid[export_cols].copy()

print('▼ Final dataset ready for export:')
print(f'  Shape: {final_export.shape}')
print(f'  Columns: {len(final_export.columns)}')
print(f'  Missing values: {final_export.isnull().sum().sum()}')

# Uncomment to actually export:
# final_export.to_csv('grid_operations_dataset.csv', index=False)
# final_export.to_excel('grid_operations_report.xlsx', index=False)

print('\n▼ Preview (first 24 hours of Zone-A):')
print(final_export[final_export['zone']=='Zone-A-Residential'].head(24).to_string(index=False))

print('\n▼ Descriptive statistics:')
print(final_export.describe().round(2).to_string())

# Duck curve for visualization
print('\n▼ Duck curve (hourly net load):')
duck_for_plot = df_grid.groupby('hour_of_day')['net_load_mw'].mean().reset_index()
print(duck_for_plot.to_string(index=False))

---

## Appendix: Pandas Technique → Grid Application Mapping

| Pandas Technique | Notebook Section | Grid Question Answered |
|---|---|---|
| `pd.DataFrame(dict)` | 1.1 | How do we structure zone infrastructure data? |
| Boolean masking (`df[df.load > x]`) | 2.1 | Which hours exceed peak demand thresholds? |
| `loc` / `iloc` | 2.1 | What is the load profile for Industrial zones? |
| Vectorized arithmetic | 2.2 | What is the net load (dispatch requirement)? |
| `isnull` / `fillna('ffill')` | 3 | How do we handle meter reading failures? |
| `pd.merge(left, right)` | 4.1 | Can we enrich load data with weather forecasts? |
| `pd.concat([df1, df2, df3])` | 4.2 | How do we stack multiple weeks for trend analysis? |
| `MultiIndex.from_product` | 4.3 | How do we model zone × week × hour panel data? |
| `groupby().agg()` | 5.1 | What is the hourly load curve across all zones? |
| `groupby().transform()` | 5.1 | How does each hour deviate from zone average? |
| `pivot_table()` | 5.2 | What does an executive zone × hour matrix look like? |
| `melt()` + `pivot()` | 5.2 | How do solar and wind generation compare by hour? |
| `.str.upper/.extract/replace` | 6 | How do we standardize messy meter IDs? |
| `DatetimeIndex` + `.rolling()` | 7 | What is the smoothed 6-hour load trend? |
| `.shift()` / `.diff()` | 7 | Can we build lag features for ML forecasting? |
| `df.eval()` | 8 | How do we efficiently compute on 50k+ meter readings? |
| `df.query()` | 8 | How do we filter high-load hot-temperature events? |

---

### Key Takeaways for Energy Sector Practitioners

1. **Duck Curve Identification** — Net load pivots reveal the solar-induced afternoon dip + evening spike that stresses conventional generation.

2. **Renewable Curtailment Detection** — When solar generation > load, excess energy must be curtailed (wasted) or stored. Pandas flags these hours automatically.

3. **Temperature Sensitivity** — Extreme heat/cold correlates with elevated load. This informs demand-response programs and capacity planning.

4. **Forecasting Feature Engineering** — Lag features (`shift()`) and rolling averages (`rolling()`) are prerequisites for ML models (XGBoost, LSTM, Prophet).

5. **Scalability** — `eval()` and `query()` enable efficient processing of millions of smart meter readings without prohibitive memory costs.

*Built with reference to the [Python Data Science Handbook](https://jakevdp.github.io/PythonDataScienceHandbook/) by Jake VanderPlas (CC-BY-NC-ND / MIT).*